In [10]:

# 1. Imports
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_text_splitters import CharacterTextSplitter

# 2. Setup Gemini API Key
GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")

llm = ChatGoogleGenerativeAI(
    temperature=0,
    model="gemini-3.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# 3. Load document and split
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.create_documents([raw_text])

# 4. Define prompts
map_prompt = PromptTemplate.from_template(
    "Summarize:\n\n{context}"
)

reduce_prompt = PromptTemplate.from_template(
    "Combine the following summaries into a single answer:\n\n{context}"
)

# 5. Create Map and Reduce chains
map_chain = map_prompt | llm
reduce_chain = reduce_prompt | llm

# 6. Convert documents
def map_docs(docs):
    return [
        {"context": doc.page_content}
        for doc in docs
    ]

# 7. Extract Gemini response text
def extract_texts(responses):
    texts = []

    for res in responses:
        if isinstance(res.content, str):
            texts.append(res.content)
        else:
            texts.append(
                "\n".join(
                    block["text"]
                    for block in res.content
                    if isinstance(block, dict) and "text" in block
                )
            )

    return {"context": "\n".join(texts)}

# 8. MapReduce chain
map_reduce_chain = (
    RunnableLambda(map_docs)
    | map_chain.map()
    | RunnableLambda(extract_texts)
    | reduce_chain
)

# 9. Execute
result = map_reduce_chain.invoke(docs)

print("Final Summary:\n")
print(result.content)


Enter Gemini API Key: ··········
Final Summary:

[{'type': 'text', 'text': 'Here is the consolidated summary:\n\n**LangChain** is a framework created by Harrison Chase for developing Large Language Model (LLM) applications—such as chatbots, document Q&A systems, and AI workflows. It enables these applications by leveraging core features including Retrieval-Augmented Generation (RAG), agents, memory, and integrated tools.', 'extras': {'signature': 'EqYNCqMNAWkUfRMyIx0sPOXjYMBHgY4zk3bY2aD1dmuwgwjq4HRBlyET5jZ4IS2W4SBsZlF6TalXA2wn/1W212A15420bT6PssJTbOTrHhoxJQ6tP3my9MuzFjXQCjKHj23E54eWI8HM5FxnNebV99P0YuIi2FH0iP8ragETXU7xjj+QtkgwwKxX+GsMJOXlt3eVZ+mh4/ueEwZVIoeqfsWAUVimcQGi0ihsDfbKebqE/dowzG93S7XCR/i8Fah3TTc3MGwmHwr+uqsD+TWKjIQSOLc0AaE2u7ENcWi2iBLV0ZTc5X0/Cqg545PES2gbJ4W489HDD2/wAgbggdvJQewdfhTxQltewtwJdDdMJ3RBe6VMX0M5OflXV5nX8VfPC5kfPkCtjCyTl+F4nlIv/O4vdRpNUKxbX7o7ttBE5fnMXEZ6FpGk64gX7WfmA2Kgi8rcnysOxuNvFttxrDXzMcKwCEuw8qjoupR482lsyK+/v1fEl5P4bKtOYnc3p94nd1jsD7BThCPJv+OBQH8FilB6GyYTZT8cu/oC

In [1]:
%pip install -q --no-deps langchain-google-genai==4.4.3

In [4]:
%pip install -q filetype

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.4.3 requires google-genai>=2.20.0, but you have google-genai 2.12.1 which is incompatible.
langchain-google-genai 4.4.3 requires langchain-core<2.0.0,>=1.7.5, but you have langchain-core 1.7.4 which is incompatible.


In [6]:

%pip install langchain-text-splitters
